In [1]:
import pandas as pd

In [2]:
isomer_df = pd.read_csv("/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/molecular_feature/isomer/all_data_isomer/isomer_pairs_stereo_only_reclassified.csv")

In [3]:
isomer_df.head()

,dataset,endpoint,toxic_smiles,nontoxic_smiles,molecular_formula,toxic_fg_full,nontoxic_fg_full,toxic_chiral_centers,nontoxic_chiral_centers,toxic_ez_bonds,...,is_position_isomer,position_different_fgs,is_fg_isomer,fg_isomer_diff,is_enantiomer,is_diastereomer,is_ez_isomer,isomer_types,primary_isomer_type,n_diff
0,clintox,clintox,C[C@@H]1C[C@H]2[C@@H]3CCC4=CC(=O)C=C[C@]4(C)[C...,C[C@H]1C[C@H]2[C@@H]3CCC4=CC(=O)C=C[C@]4(C)[C@...,C22H29FO5,{},{},"[{'atom_idx': 1, 'config': 'S'}, {'atom_idx': ...","[{'atom_idx': 1, 'config': 'R'}, {'atom_idx': ...",[],...,False,[],False,{},False,True,False,['Diastereomer'],Diastereomer,1
1,clintox,clintox,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,C53H83NO14,{},{},"[{'atom_idx': 2, 'config': 'R'}, {'atom_idx': ...","[{'atom_idx': 2, 'config': 'R'}, {'atom_idx': ...","[{'bond': (46, 47), 'geometry': 'E'}, {'bond':...",...,False,[],False,{},False,True,True,"['Diastereomer', 'E/Z Isomer']",Diastereomer,1
2,dilist,dilist,COc1cccc2C(=O)c3c(O)c4C[C@](O)(C[C@H](O[C@H]5C...,COc1cccc2C(=O)c3c(O)c4C[C@](O)(C[C@H](O[C@H]5C...,C27H29NO11,{},{},"[{'atom_idx': 14, 'config': 'S'}, {'atom_idx':...","[{'atom_idx': 14, 'config': 'S'}, {'atom_idx':...",[],...,False,[],False,{},False,True,False,['Diastereomer'],Diastereomer,1
3,herg,herg,C[C@@H](O)c1cn(-c2ccc(F)cc2)c2ccc(Cl)cc12,C[C@H](O)c1cn(-c2ccc(F)cc2)c2ccc(Cl)cc12,C16H13ClFNO,{},{},"[{'atom_idx': 1, 'config': 'R'}]","[{'atom_idx': 1, 'config': 'S'}]",[],...,False,[],False,{},True,False,False,['Enantiomer'],Enantiomer,1
4,herg,herg,CC[C@@H](O)c1cn(-c2ccc(F)cc2)c2ccc(Cl)cc12,CC[C@H](O)c1cn(-c2ccc(F)cc2)c2ccc(Cl)cc12,C17H15ClFNO,{},{},"[{'atom_idx': 2, 'config': 'R'}]","[{'atom_idx': 2, 'config': 'S'}]",[],...,False,[],False,{},True,False,False,['Enantiomer'],Enantiomer,1


In [4]:
for isomer_type in isomer_df['primary_isomer_type'].unique():
    print(isomer_type)
    df = isomer_df[isomer_df['primary_isomer_type'] == isomer_type]
    print(df['n_diff'].value_counts())
    print(df['n_diff'].describe())

Diastereomer
n_diff
1     26
2      5
3      3
10     1
11     1
4      1
Name: count, dtype: int64
count    37.000000
mean      1.891892
std       2.220908
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max      11.000000
Name: n_diff, dtype: float64
Enantiomer
n_diff
2    29
1    26
3     7
5     1
Name: count, dtype: int64
count    63.000000
mean      1.746032
std       0.782227
min       1.000000
25%       1.000000
50%       2.000000
75%       2.000000
max       5.000000
Name: n_diff, dtype: float64
E/Z Isomer
n_diff
1    29
Name: count, dtype: int64
count    29.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
Name: n_diff, dtype: float64


## `n_diff` 정의 (출처: `molecular_feature/isomer/src/find_isomer.py`)

`n_diff`는 **입체 화학이 다른 atom(chiral center) 또는 bond(E/Z)의 개수**다.

- **Chiral**: 같은 atom_idx에서 config(R/S)가 다르면 1개씩. 한쪽에만 있는 center도 1.
- **E/Z**: 같은 bond에서 geometry(E/Z)가 다르면 1개씩. 한쪽에만 있는 bond도 1.
- **n_diff** = (config가 다른 chiral center 개수) + (geometry가 다른 E/Z bond 개수)

예: toxic 2R·1S, nontoxic 1R·2S (동일 3개 center, 전부 R↔S 반대) → **n_diff = 3**.  
예: E/Z bond 1쌍만 E vs Z로 다름 → **n_diff = 1** (bond 1개).

---

## `n_diff = 0`이 나오는 경우

**주의:** Enantiomer는 “모두 반대”이지만, **n_diff는 R/S 개수 차이의 합**이라서 Enantiomer라도 n_diff가 0이 아닐 수 있다.  
toxic (R, S) = (a, b), nontoxic = (b, a)일 때 `chiral_diff = 2|a−b|` 이므로, **a = b일 때만** 0이 된다.

1. **Enantiomer (7건만 n_diff=0, 나머지는 2 또는 4)**  
   - Enantiomer는 모든 chiral center가 R↔S로만 반대다.  
   - 이때 toxic (R, S) = (a, b), nontoxic = (b, a)이면  
     `chiral_diff = |a−b| + |b−a| = 2|a−b|`.  
   - **a = b** (R 개수와 S 개수가 같음)일 때만 chiral_diff = 0.  
     예: 2R·2S vs 2S·2R → 0.  
   - **a ≠ b**이면 n_diff > 0. 예: 3R·1S vs 1R·3S → 2|3−1| = **4**.  
   - 따라서 Enantiomer 63건 중 7건만 n_diff=0이고, 나머지는 2 또는 4.

2. **Diastereomer (3건)**  
   - **chiral_centers / ez_bonds 정보가 비어 있거나 파싱 실패**인 경우,  
     R/S/E/Z가 모두 0으로 들어가서 `chiral_diff = 0`, `ez_diff = 0` → **n_diff = 0**.  
   - 또는 **재분류 시 예외 발생** 시(`compare_canonical_no_stereo.py`에서 `classify_isomer_type` 실패)  
     기본값으로 `n_diff = 0`이 들어간다.

3. **E/Z Isomer**  
   - 현재 데이터에서는 E/Z만 다른 pair는 모두 E/Z bond가 1쌍만 다르게 나와서 `n_diff = 2`만 존재하고, 0은 없다.